# Banking77 Intent Classification with Unsloth

**Course:** CSC15012 – Applications of Natural Language Processing in Industry  
**Student:** Lê Đình Minh Quân – 23127460  
**University:** HCMUS – VNUHCM

---

### Autopilot Notebook – Run all cells from top to bottom!

| Step | Description |
|------|-------------|
| 1 | Install dependencies |
| 2 | Create project structure & files |
| 3 | Preprocess BANKING77 dataset |
| 4 | Fine-tune LLaMA 3.2 3B (QLoRA via Unsloth) |
| 5 | Evaluate & demo inference |
| 6 | Push to GitHub |

> **Runtime:** Use a **GPU** runtime (H100 / A100 / T4). Go to `Runtime → Change runtime type → GPU`.  
> **Secret:** Store your GitHub PAT as **`GITHUB_PAT`** in Colab Secrets (key icon, left sidebar).


In [22]:
# ============================================================
# Step 0: Install dependencies, set HF token & verify GPU
# ============================================================
!nvidia-smi
print()

!pip install -q unsloth
!pip install -q datasets pandas scikit-learn pyyaml

# Set HuggingFace token (avoids rate-limit warnings)
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Colab Secrets')
except Exception:
    print('No HF_TOKEN found (optional – public datasets work without it)')

import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("\nDependencies installed!")


Mon Apr 13 05:46:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   32C    P0            117W /  700W |     719MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [23]:
# ============================================================
# Step 1: Create project structure & write config/utility files
# ============================================================
import os

PROJECT = '/content/banking-intent-unsloth'
for d in ['scripts', 'configs', 'sample_data']:
    os.makedirs(f'{PROJECT}/{d}', exist_ok=True)

%cd /content/banking-intent-unsloth

with open('configs/train.yaml', 'w') as _f:
    _f.write('# ==============================================================================\n# Training Configuration for Banking77 Intent Classification with Unsloth\n# ==============================================================================\n\nmodel:\n  name: "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"\n  max_seq_length: 512\n  load_in_4bit: true\n  dtype: null  # auto-detect (float16 for older GPUs, bfloat16 for Ampere+)\n\nlora:\n  r: 16\n  lora_alpha: 16\n  lora_dropout: 0.0\n  target_modules:\n    - q_proj\n    - k_proj\n    - v_proj\n    - o_proj\n    - gate_proj\n    - up_proj\n    - down_proj\n  bias: "none"\n  use_gradient_checkpointing: "unsloth"\n  random_state: 3407\n\ntraining:\n  per_device_train_batch_size: 8\n  gradient_accumulation_steps: 4\n  warmup_steps: 10\n  num_train_epochs: 3\n  max_steps: -1  # set to positive number to override num_train_epochs\n  learning_rate: 2.0e-4\n  weight_decay: 0.01\n  fp16: false\n  bf16: true\n  logging_steps: 10\n  optim: "adamw_8bit"\n  lr_scheduler_type: "linear"\n  seed: 42\n  output_dir: "outputs"\n  save_strategy: "epoch"\n  report_to: "none"\n\ndata:\n  train_file: "sample_data/train.csv"\n  test_file: "sample_data/test.csv"\n  label_map_file: "sample_data/label_map.json"\n\nsave:\n  model_dir: "saved_model"\n\nevaluate:\n  run_eval_after_training: true\n  max_new_tokens: 64\n  temperature: 0.0\n  results_file: "evaluation_results.json"\n')
print('Written: configs/train.yaml')

with open('configs/inference.yaml', 'w') as _f:
    _f.write('# ==============================================================================\n# Inference Configuration for Banking77 Intent Classification\n# ==============================================================================\n\nmodel:\n  name: "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"\n  checkpoint_path: "saved_model"\n  max_seq_length: 512\n  load_in_4bit: true\n  dtype: null\n\ndata:\n  label_map_file: "sample_data/label_map.json"\n  test_file: "sample_data/test.csv"\n\ngeneration:\n  max_new_tokens: 64\n  temperature: 0.0\n  do_sample: false\n')
print('Written: configs/inference.yaml')

with open('train.sh', 'w') as _f:
    _f.write('#!/usr/bin/env bash\n# ==============================================================================\n# train.sh – End-to-end pipeline: preprocess data → fine-tune model\n# ==============================================================================\nset -euo pipefail\n\necho "=============================================="\necho " Banking77 Intent Classification – Training"\necho "=============================================="\n\n# Step 1: Preprocess data\necho ""\necho "[Step 1/2] Preprocessing BANKING77 dataset …"\npython scripts/preprocess_data.py \\\n    --train_samples_per_class 40 \\\n    --test_samples_per_class 10 \\\n    --output_dir sample_data \\\n    --seed 42\n\n# Step 2: Fine-tune with Unsloth\necho ""\necho "[Step 2/2] Fine-tuning model with Unsloth …"\npython scripts/train.py --config configs/train.yaml\n\necho ""\necho "=============================================="\necho " Training pipeline complete!"\necho "=============================================="\n')
print('Written: train.sh')

with open('inference.sh', 'w') as _f:
    _f.write('#!/usr/bin/env bash\n# ==============================================================================\n# inference.sh – Run intent classification inference\n# ==============================================================================\nset -euo pipefail\n\nCONFIG="configs/inference.yaml"\n\necho "=============================================="\necho " Banking77 Intent Classification – Inference"\necho "=============================================="\n\n# --- Option 1: Single message (default demo) ---\nif [ $# -ge 1 ]; then\n    MESSAGE="$*"\n    echo ""\n    echo "Classifying message: \\"${MESSAGE}\\""\n    echo ""\n    python scripts/inference.py --config "$CONFIG" --message "$MESSAGE"\nelse\n    # Demo with several example messages\n    echo ""\n    echo "Running demo predictions …"\n    echo ""\n\n    python scripts/inference.py --config "$CONFIG" \\\n        --message "I want to activate my new card"\n\n    echo "---"\n    python scripts/inference.py --config "$CONFIG" \\\n        --message "Why was I charged an extra fee on my statement?"\n\n    echo "---"\n    python scripts/inference.py --config "$CONFIG" \\\n        --message "My card payment is still pending after 3 days"\n\n    echo "---"\n    python scripts/inference.py --config "$CONFIG" \\\n        --message "How do I change my PIN number?"\n\n    echo "---"\n    python scripts/inference.py --config "$CONFIG" \\\n        --message "I lost my phone and need to secure my account"\n\n    # Also run full test-set evaluation\n    echo ""\n    echo "=============================================="\n    echo " Running evaluation on test set …"\n    echo "=============================================="\n    python scripts/inference.py --config "$CONFIG" --evaluate\nfi\n\necho ""\necho "=============================================="\necho " Inference complete!"\necho "=============================================="\n')
print('Written: inference.sh')

with open('requirements.txt', 'w') as _f:
    _f.write('# ==============================================================================\n# Banking77 Intent Classification – Dependencies\n# ==============================================================================\n# NOTE: On Google Colab, install unsloth first with:\n#   pip install unsloth\n# Then install the remaining dependencies below.\n# ==============================================================================\n\nunsloth\ntorch>=2.1.0\ntransformers>=4.40.0\ndatasets>=2.18.0\ntrl>=0.8.0\npeft>=0.10.0\nbitsandbytes>=0.43.0\naccelerate>=0.28.0\npandas>=2.0.0\nscikit-learn>=1.3.0\npyyaml>=6.0\nxformers\n')
print('Written: requirements.txt')

with open('.gitignore', 'w') as _f:
    _f.write('# Python\n__pycache__/\n*.py[cod]\n*.egg-info/\n\n# Model checkpoints & outputs (large files – do NOT push to GitHub)\nsaved_model/\noutputs/\n*.bin\n*.safetensors\n\n# Evaluation results (regenerated on each run)\nevaluation_results.json\n\n# IDE / OS\n.vscode/\n.idea/\n.DS_Store\nThumbs.db\n\n# Jupyter\n.ipynb_checkpoints/\n')
print('Written: .gitignore')

with open('README.md', 'w') as _f:
    _f.write('# Banking77 Intent Classification with Unsloth\n\nFine-tuning a large language model (LLaMA 3.2 3B) for **banking intent classification** on the [BANKING77](https://huggingface.co/datasets/PolyAI/banking77) dataset using **Unsloth** (4-bit QLoRA).\n\n> **Course**: CSC15012 – Applications of Natural Language Processing in Industry  \n> **Student**: Lê Đình Minh Quân – 23127460  \n> **University**: HCMUS – VNUHCM\n\n---\n\n## Project Structure\n\n```\nbanking-intent-unsloth/\n├── scripts/\n│   ├── preprocess_data.py    # Download & preprocess BANKING77\n│   ├── train.py              # Fine-tune with Unsloth + LoRA\n│   └── inference.py          # Standalone inference class\n├── configs/\n│   ├── train.yaml            # Training hyperparameters\n│   └── inference.yaml        # Inference configuration\n├── sample_data/\n│   ├── train.csv             # Sampled training set\n│   ├── test.csv              # Sampled test set\n│   └── label_map.json        # Intent label mapping\n├── train.sh                  # One-click training script\n├── inference.sh              # One-click inference / demo script\n├── requirements.txt          # Python dependencies\n├── .gitignore\n└── README.md                 # This file\n```\n\n---\n\n## Quick Start\n\n### 1. Environment Setup (Google Colab – recommended)\n\nOpen a **Colab notebook** with a **GPU runtime** (T4 / A100 / H100) and run:\n\n```bash\n# Install Unsloth (optimized for Colab)\npip install unsloth\n\n# Install remaining dependencies\npip install datasets pandas scikit-learn pyyaml\n\n# Clone this repository\ngit clone https://github.com/ledinhminhquan/banking-intent-unsloth.git\ncd banking-intent-unsloth\n```\n\n### 2. Data Preparation\n\n```bash\npython scripts/preprocess_data.py \\\n    --train_samples_per_class 40 \\\n    --test_samples_per_class 10 \\\n    --output_dir sample_data \\\n    --seed 42\n```\n\nThis downloads the BANKING77 dataset, samples a subset (≈ 3,080 train / 770 test), applies text preprocessing, and saves CSV files under `sample_data/`.\n\n### 3. Training\n\n```bash\npython scripts/train.py --config configs/train.yaml\n```\n\nOr use the all-in-one script:\n\n```bash\nbash train.sh\n```\n\n**Key hyperparameters** (editable in `configs/train.yaml`):\n\n| Parameter | Value |\n|---|---|\n| Base model | `unsloth/Llama-3.2-3B-Instruct-bnb-4bit` |\n| LoRA rank (r) | 16 |\n| LoRA alpha | 16 |\n| Learning rate | 2 × 10⁻⁴ |\n| Optimizer | AdamW 8-bit |\n| Batch size | 8 × 4 (gradient accumulation) |\n| Epochs | 3 |\n| Max sequence length | 512 |\n| Precision | bf16 (auto-detects fp16 fallback) |\n\nThe fine-tuned LoRA adapter is saved to `saved_model/`.\n\n### 4. Inference\n\n**Single message:**\n```bash\npython scripts/inference.py \\\n    --config configs/inference.yaml \\\n    --message "I want to activate my new card"\n```\n\n**Test-set evaluation:**\n```bash\npython scripts/inference.py --config configs/inference.yaml --evaluate\n```\n\n**Interactive mode:**\n```bash\npython scripts/inference.py --config configs/inference.yaml --interactive\n```\n\n**Or use the demo script:**\n```bash\nbash inference.sh\n```\n\n### 5. Inference Class API\n\n```python\nfrom scripts.inference import IntentClassification\n\nclassifier = IntentClassification("configs/inference.yaml")\n\nlabel = classifier("I need to change my PIN")\nprint(label)  # e.g. "change_pin"\n```\n\n---\n\n## Results\n\n<!-- Update these numbers after training -->\n| Metric | Value |\n|---|---|\n| Test Accuracy | _TBD_ |\n| Training Loss | _TBD_ |\n| Training Time | _TBD_ |\n\nFull classification report is saved to `evaluation_results.json` after training.\n\n---\n\n## Video Demonstration\n\n<!-- Replace with your actual Google Drive link -->\n📹 **Video link**: [Google Drive – Demo Video](https://drive.google.com/YOUR_VIDEO_LINK_HERE)\n\nThe video shows:\n1. Running the inference script\n2. Example input messages and predicted intents\n3. Test-set accuracy\n\n---\n\n## References\n\n- [BANKING77 Dataset](https://huggingface.co/datasets/PolyAI/banking77) – Casanueva et al. (2020)\n- [Unsloth](https://github.com/unslothai/unsloth) – Efficient LLM fine-tuning\n- [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685) – Hu et al. (2021)\n\n---\n\n## License\n\nThis project is for educational purposes (CSC15012 – HCMUS).\n')
print('Written: README.md')

print('\nAll config and utility files created!')

/content/banking-intent-unsloth
Written: configs/train.yaml
Written: configs/inference.yaml
Written: train.sh
Written: inference.sh
Written: requirements.txt
Written: .gitignore
Written: README.md

All config and utility files created!


## Step 2: Create Python Scripts

The following cells write the three main scripts to disk.

In [24]:
%%writefile scripts/preprocess_data.py
"""
preprocess_data.py
------------------
Download the BANKING77 dataset from HuggingFace, sample a manageable subset,
perform basic text preprocessing, and save train/test splits plus a label map.
"""

import os
import json
import argparse
import re

import pandas as pd
from datasets import load_dataset

BANKING77_LABELS = [
    "activate_my_card", "age_limit", "apple_pay_or_google_pay",
    "atm_support", "automatic_top_up",
    "balance_not_updated_after_bank_transfer",
    "balance_not_updated_after_cheque_or_cash_deposit",
    "beneficiary_not_allowed", "cancel_transfer", "card_about_to_expire",
    "card_acceptance", "card_arrival", "card_delivery_estimate",
    "card_linking", "card_not_working", "card_payment_fee_charged",
    "card_payment_not_recognised", "card_payment_wrong_exchange_rate",
    "card_swallowed", "cash_withdrawal_charge",
    "cash_withdrawal_not_recognised", "change_pin", "compromised_card",
    "contactless_not_working", "country_support", "declined_card_payment",
    "declined_cash_withdrawal", "declined_transfer",
    "direct_debit_payment_not_recognised", "disposable_card_limits",
    "edit_personal_details", "exchange_charge", "exchange_rate",
    "exchange_via_app", "extra_charge_on_statement", "failed_transfer",
    "fiat_currency_support", "freeze_card",
    "get_disposable_virtual_card", "get_physical_card",
    "getting_spare_card", "getting_virtual_card", "lost_or_stolen_card",
    "lost_or_stolen_phone", "order_physical_card", "passcode_forgotten",
    "pending_card_payment", "pending_cash_withdrawal", "pending_top_up",
    "pending_transfer", "pin_blocked", "receiving_money",
    "Refund_not_showing_up", "request_refund", "reverted_card_payment?",
    "supported_cards_and_currencies", "terminate_account",
    "top_up_by_bank_transfer_charge", "top_up_by_card_charge",
    "top_up_by_cash_or_cheque", "top_up_failed", "top_up_limits",
    "top_up_reverted", "topping_up_by_card", "transaction_charged_twice",
    "transfer_fee_charged", "transfer_into_account",
    "transfer_not_received_by_recipient", "transfer_timing",
    "unable_to_verify_identity", "verify_my_identity",
    "verify_source_of_funds", "verify_top_up",
    "virtual_card_not_working", "visa_or_mastercard",
    "why_verify_identity", "wrong_amount_of_cash_received",
    "wrong_exchange_rate_for_cash_withdrawal",
]


def _load_banking77():
    """Load BANKING77 with multiple fallbacks for different datasets versions."""
    # Strategy 1: direct load (works with datasets < 3.0)
    try:
        ds = load_dataset("PolyAI/banking77")
        label_names = ds["train"].features["label"].names
        print("  Loaded via default method.")
        return ds, label_names
    except Exception as e:
        print(f"  Default load failed: {e}")

    # Strategy 2: auto-converted parquet branch (works with datasets >= 3.0)
    try:
        ds = load_dataset("PolyAI/banking77", revision="refs/convert/parquet")
        try:
            label_names = ds["train"].features["label"].names
        except (AttributeError, KeyError):
            label_names = BANKING77_LABELS
        print("  Loaded via parquet revision.")
        return ds, label_names
    except Exception as e:
        print(f"  Parquet revision failed: {e}")

    # Strategy 3: load parquet files directly from HF Hub
    try:
        base = "hf://datasets/PolyAI/banking77@refs%2Fconvert%2Fparquet"
        ds = load_dataset(
            "parquet",
            data_files={
                "train": f"{base}/default/train/0000.parquet",
                "test": f"{base}/default/test/0000.parquet",
            },
        )
        print("  Loaded via direct parquet URLs.")
        return ds, BANKING77_LABELS
    except Exception as e:
        print(f"  Direct parquet failed: {e}")

    raise RuntimeError(
        "Could not load BANKING77. Try: pip install 'datasets<3'"
    )


def clean_text(text: str) -> str:
    """Basic text normalization for banking queries."""
    text = text.strip()
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text


def main(args):
    print("Loading BANKING77 dataset from HuggingFace …")
    dataset, label_names = _load_banking77()

    id2label = {str(i): name for i, name in enumerate(label_names)}
    label2id = {name: i for i, name in enumerate(label_names)}
    num_labels = len(label_names)

    train_df = pd.DataFrame(dataset["train"])
    test_df = pd.DataFrame(dataset["test"])

    train_df["label_text"] = train_df["label"].map(lambda x: id2label[str(x)])
    test_df["label_text"] = test_df["label"].map(lambda x: id2label[str(x)])

    # ---- Sample a subset to fit available compute ----
    if args.train_samples_per_class > 0:
        train_df = (
            train_df.groupby("label", group_keys=False)
            .apply(
                lambda g: g.sample(
                    n=min(args.train_samples_per_class, len(g)),
                    random_state=args.seed,
                )
            )
            .reset_index(drop=True)
        )

    if args.test_samples_per_class > 0:
        test_df = (
            test_df.groupby("label", group_keys=False)
            .apply(
                lambda g: g.sample(
                    n=min(args.test_samples_per_class, len(g)),
                    random_state=args.seed,
                )
            )
            .reset_index(drop=True)
        )

    # ---- Clean text ----
    train_df["text"] = train_df["text"].apply(clean_text)
    test_df["text"] = test_df["text"].apply(clean_text)

    # ---- Shuffle ----
    train_df = train_df.sample(frac=1, random_state=args.seed).reset_index(drop=True)
    test_df = test_df.sample(frac=1, random_state=args.seed).reset_index(drop=True)

    # ---- Save ----
    os.makedirs(args.output_dir, exist_ok=True)

    train_path = os.path.join(args.output_dir, "train.csv")
    test_path = os.path.join(args.output_dir, "test.csv")
    label_map_path = os.path.join(args.output_dir, "label_map.json")

    train_df[["text", "label", "label_text"]].to_csv(train_path, index=False)
    test_df[["text", "label", "label_text"]].to_csv(test_path, index=False)

    with open(label_map_path, "w", encoding="utf-8") as f:
        json.dump(
            {"id2label": id2label, "label2id": label2id, "num_labels": num_labels},
            f,
            indent=2,
            ensure_ascii=False,
        )

    print(f"Number of intent labels : {num_labels}")
    print(f"Training samples        : {len(train_df)}")
    print(f"Test samples            : {len(test_df)}")
    print(f"Files saved to          : {args.output_dir}/")
    print(f"  - {train_path}")
    print(f"  - {test_path}")
    print(f"  - {label_map_path}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Preprocess BANKING77 dataset for intent classification"
    )
    parser.add_argument(
        "--train_samples_per_class",
        type=int,
        default=40,
        help="Max training samples per intent class (0 = use all)",
    )
    parser.add_argument(
        "--test_samples_per_class",
        type=int,
        default=10,
        help="Max test samples per intent class (0 = use all)",
    )
    parser.add_argument(
        "--output_dir",
        type=str,
        default="sample_data",
        help="Directory to save processed CSV files",
    )
    parser.add_argument("--seed", type=int, default=42, help="Random seed")
    args = parser.parse_args()
    main(args)


Overwriting scripts/preprocess_data.py


In [25]:
%%writefile scripts/train.py
"""
train.py
--------
Fine-tune a pre-trained LLM for banking intent classification using Unsloth
(4-bit LoRA) and the HuggingFace SFTTrainer.
"""

import os
import json
import argparse
import time
from difflib import get_close_matches

import yaml
import torch
import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig
from sklearn.metrics import accuracy_score, classification_report

# ---------------------------------------------------------------------------
# Prompt template (Alpaca-style) – must be identical at training & inference
# ---------------------------------------------------------------------------
PROMPT_TEMPLATE = (
    "Below is an instruction that describes a task, paired with an input "
    "that provides further context. Write a response that appropriately "
    "completes the request.\n\n"
    "### Instruction:\n"
    "Classify the intent of the following banking customer message. "
    "Reply with only the intent label, nothing else.\n\n"
    "### Input:\n{text}\n\n"
    "### Response:\n{label}"
)

PROMPT_TEMPLATE_INFERENCE = (
    "Below is an instruction that describes a task, paired with an input "
    "that provides further context. Write a response that appropriately "
    "completes the request.\n\n"
    "### Instruction:\n"
    "Classify the intent of the following banking customer message. "
    "Reply with only the intent label, nothing else.\n\n"
    "### Input:\n{text}\n\n"
    "### Response:\n"
)


def format_dataset(df: pd.DataFrame, tokenizer) -> Dataset:
    """Convert a DataFrame into a HuggingFace Dataset with formatted prompts."""
    eos = tokenizer.eos_token
    records = []
    for _, row in df.iterrows():
        prompt = PROMPT_TEMPLATE.format(text=row["text"], label=row["label_text"])
        records.append({"formatted_text": prompt + eos})
    return Dataset.from_pandas(pd.DataFrame(records))


def evaluate_model(model, tokenizer, test_df, id2label, gen_cfg):
    """Run inference on the test set and return accuracy + classification report."""
    FastLanguageModel.for_inference(model)

    all_labels = list(set(id2label.values()))
    y_true, y_pred = [], []

    print(f"\nEvaluating on {len(test_df)} test samples …")
    start = time.time()

    for idx, row in test_df.iterrows():
        prompt = PROMPT_TEMPLATE_INFERENCE.format(text=row["text"])
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=gen_cfg.get("max_new_tokens", 64),
                temperature=gen_cfg.get("temperature", 0.0),
                do_sample=False,
                use_cache=True,
            )

        new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        predicted = response.split("\n")[0].strip()

        if predicted not in all_labels:
            matches = get_close_matches(predicted, all_labels, n=1, cutoff=0.4)
            predicted = matches[0] if matches else predicted

        y_true.append(row["label_text"])
        y_pred.append(predicted)

        if (idx + 1) % 50 == 0:
            elapsed = time.time() - start
            print(f"  [{idx+1}/{len(test_df)}]  elapsed {elapsed:.1f}s")

    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, zero_division=0)
    elapsed = time.time() - start

    print(f"\nTest Accuracy : {acc:.4f}  ({sum(a==b for a,b in zip(y_true,y_pred))}/{len(y_true)})")
    print(f"Evaluation time: {elapsed:.1f}s")
    print(f"\nClassification Report:\n{report}")

    return {
        "accuracy": acc,
        "num_correct": sum(a == b for a, b in zip(y_true, y_pred)),
        "num_total": len(y_true),
        "classification_report": report,
        "predictions": [
            {"text": t, "true": yt, "pred": yp}
            for t, yt, yp in zip(test_df["text"].tolist(), y_true, y_pred)
        ],
    }


def main(config_path: str):
    # ---- Load config ----
    with open(config_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    model_cfg = cfg["model"]
    lora_cfg = cfg["lora"]
    train_cfg = cfg["training"]
    data_cfg = cfg["data"]
    save_cfg = cfg["save"]
    eval_cfg = cfg.get("evaluate", {})

    # ---- Load preprocessed data ----
    train_df = pd.read_csv(data_cfg["train_file"])
    test_df = pd.read_csv(data_cfg["test_file"])

    with open(data_cfg["label_map_file"], encoding="utf-8") as f:
        label_data = json.load(f)
    id2label = label_data["id2label"]

    print(f"Training samples : {len(train_df)}")
    print(f"Test samples     : {len(test_df)}")
    print(f"Num labels       : {label_data['num_labels']}")

    # ---- Load model ----
    print(f"\nLoading model: {model_cfg['name']} …")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_cfg["name"],
        max_seq_length=model_cfg["max_seq_length"],
        load_in_4bit=model_cfg["load_in_4bit"],
        dtype=model_cfg.get("dtype"),
    )

    # ---- Apply LoRA adapters ----
    model = FastLanguageModel.get_peft_model(
        model,
        r=lora_cfg["r"],
        target_modules=lora_cfg["target_modules"],
        lora_alpha=lora_cfg["lora_alpha"],
        lora_dropout=lora_cfg["lora_dropout"],
        bias=lora_cfg["bias"],
        use_gradient_checkpointing=lora_cfg["use_gradient_checkpointing"],
        random_state=lora_cfg.get("random_state", 3407),
    )

    # ---- Format training dataset ----
    train_dataset = format_dataset(train_df, tokenizer)
    print(f"Formatted training dataset: {len(train_dataset)} examples")

    # ---- Training arguments (SFTConfig = TrainingArguments + SFT params) ----
    use_bf16 = is_bfloat16_supported()
    sft_config = SFTConfig(
        per_device_train_batch_size=train_cfg["per_device_train_batch_size"],
        gradient_accumulation_steps=train_cfg["gradient_accumulation_steps"],
        warmup_steps=train_cfg["warmup_steps"],
        num_train_epochs=train_cfg["num_train_epochs"],
        max_steps=train_cfg["max_steps"],
        learning_rate=train_cfg["learning_rate"],
        weight_decay=train_cfg["weight_decay"],
        fp16=not use_bf16,
        bf16=use_bf16,
        logging_steps=train_cfg["logging_steps"],
        optim=train_cfg["optim"],
        lr_scheduler_type=train_cfg["lr_scheduler_type"],
        seed=train_cfg["seed"],
        output_dir=train_cfg["output_dir"],
        save_strategy=train_cfg["save_strategy"],
        report_to=train_cfg.get("report_to", "none"),
        dataset_text_field="formatted_text",
        max_seq_length=model_cfg["max_seq_length"],
        dataset_num_proc=2,
        packing=False,
    )

    # ---- Trainer ----
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        args=sft_config,
    )

    # ---- Train ----
    print("\n========== Starting training ==========")
    train_start = time.time()
    trainer_stats = trainer.train()
    train_elapsed = time.time() - train_start
    print(f"Training completed in {train_elapsed:.1f}s")
    print(f"Training loss: {trainer_stats.training_loss:.4f}")

    # ---- Save model ----
    save_dir = save_cfg["model_dir"]
    os.makedirs(save_dir, exist_ok=True)
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f"\nModel saved to: {save_dir}/")

    # ---- Evaluate ----
    if eval_cfg.get("run_eval_after_training", True):
        gen_cfg = {
            "max_new_tokens": eval_cfg.get("max_new_tokens", 64),
            "temperature": eval_cfg.get("temperature", 0.0),
        }
        results = evaluate_model(model, tokenizer, test_df, id2label, gen_cfg)

        results_file = eval_cfg.get("results_file", "evaluation_results.json")
        with open(results_file, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "accuracy": results["accuracy"],
                    "num_correct": results["num_correct"],
                    "num_total": results["num_total"],
                    "classification_report": results["classification_report"],
                    "training_loss": trainer_stats.training_loss,
                    "training_time_seconds": train_elapsed,
                },
                f,
                indent=2,
                ensure_ascii=False,
            )
        print(f"\nEvaluation results saved to: {results_file}")

    print("\n========== Done ==========")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Fine-tune model with Unsloth")
    parser.add_argument(
        "--config",
        type=str,
        default="configs/train.yaml",
        help="Path to training config YAML",
    )
    args = parser.parse_args()
    main(args.config)


Overwriting scripts/train.py


In [26]:
%%writefile scripts/inference.py
"""
inference.py
------------
Standalone inference module for banking intent classification.
Loads a fine-tuned Unsloth/LoRA checkpoint and predicts the intent of a
single text input.

Usage examples:
    # Single message
    python scripts/inference.py --config configs/inference.yaml \
        --message "I need to activate my new card"

    # Evaluate full test set
    python scripts/inference.py --config configs/inference.yaml --evaluate

    # Interactive mode
    python scripts/inference.py --config configs/inference.yaml --interactive
"""

import os
import json
import argparse
import time
from difflib import get_close_matches

import yaml
import torch
import pandas as pd
from unsloth import FastLanguageModel
from sklearn.metrics import accuracy_score, classification_report

# Must match the template used during training (see scripts/train.py)
PROMPT_TEMPLATE = (
    "Below is an instruction that describes a task, paired with an input "
    "that provides further context. Write a response that appropriately "
    "completes the request.\n\n"
    "### Instruction:\n"
    "Classify the intent of the following banking customer message. "
    "Reply with only the intent label, nothing else.\n\n"
    "### Input:\n{text}\n\n"
    "### Response:\n"
)


class IntentClassification:
    """
    Banking intent classifier that wraps a fine-tuned Unsloth model.

    Parameters
    ----------
    model_path : str
        Path to a YAML configuration file that contains at least the path to
        the saved model checkpoint, label map, and generation settings.
    """

    def __init__(self, model_path: str):
        with open(model_path, encoding="utf-8") as f:
            config = yaml.safe_load(f)

        model_cfg = config["model"]
        data_cfg = config["data"]
        self.gen_cfg = config.get("generation", {})

        checkpoint = model_cfg["checkpoint_path"]
        print(f"Loading model from: {checkpoint} …")
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=checkpoint,
            max_seq_length=model_cfg.get("max_seq_length", 512),
            load_in_4bit=model_cfg.get("load_in_4bit", True),
            dtype=model_cfg.get("dtype"),
        )
        FastLanguageModel.for_inference(self.model)

        with open(data_cfg["label_map_file"], encoding="utf-8") as f:
            label_data = json.load(f)

        self.id2label = label_data["id2label"]
        self.label2id = label_data["label2id"]
        self.all_labels = list(self.label2id.keys())
        print(f"Loaded {len(self.all_labels)} intent labels.")

    def __call__(self, message: str) -> str:
        """
        Predict the intent label of a banking customer message.

        Parameters
        ----------
        message : str
            A single banking customer query.

        Returns
        -------
        str
            The predicted intent label.
        """
        prompt = PROMPT_TEMPLATE.format(text=message.strip().lower())
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.gen_cfg.get("max_new_tokens", 64),
                temperature=self.gen_cfg.get("temperature", 0.0),
                do_sample=False,
                use_cache=True,
            )

        new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
        response = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        predicted_label = response.split("\n")[0].strip()

        if predicted_label not in self.all_labels:
            matches = get_close_matches(
                predicted_label, self.all_labels, n=1, cutoff=0.4
            )
            if matches:
                predicted_label = matches[0]

        return predicted_label


def evaluate(classifier: IntentClassification, test_file: str):
    """Evaluate the classifier on the full test set."""
    test_df = pd.read_csv(test_file)
    y_true, y_pred = [], []

    print(f"\nRunning evaluation on {len(test_df)} test samples …\n")
    start = time.time()

    for idx, row in test_df.iterrows():
        pred = classifier(row["text"])
        y_true.append(row["label_text"])
        y_pred.append(pred)

        if (idx + 1) % 50 == 0:
            acc_so_far = accuracy_score(y_true, y_pred)
            elapsed = time.time() - start
            print(f"  [{idx+1}/{len(test_df)}]  acc={acc_so_far:.4f}  elapsed={elapsed:.1f}s")

    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, zero_division=0)

    print(f"\n{'='*60}")
    print(f"Test Accuracy: {acc:.4f}  ({sum(a==b for a,b in zip(y_true,y_pred))}/{len(y_true)})")
    print(f"{'='*60}")
    print(f"\nClassification Report:\n{report}")
    return acc


def interactive_mode(classifier: IntentClassification):
    """Run an interactive loop for manual testing."""
    print("\n=== Interactive Intent Classification ===")
    print("Type a banking message and press Enter. Type 'quit' to exit.\n")
    while True:
        try:
            message = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if message.lower() in ("quit", "exit", "q"):
            break
        if not message:
            continue
        label = classifier(message)
        print(f"  → Intent: {label}\n")
    print("Goodbye!")


def main():
    parser = argparse.ArgumentParser(
        description="Banking intent classification – inference"
    )
    parser.add_argument(
        "--config",
        type=str,
        default="configs/inference.yaml",
        help="Path to inference config YAML",
    )
    parser.add_argument(
        "--message",
        type=str,
        default=None,
        help="A single message to classify",
    )
    parser.add_argument(
        "--evaluate",
        action="store_true",
        help="Run evaluation on the test set",
    )
    parser.add_argument(
        "--interactive",
        action="store_true",
        help="Start interactive classification mode",
    )
    args = parser.parse_args()

    classifier = IntentClassification(args.config)

    if args.message:
        label = classifier(args.message)
        print(f"\nMessage : {args.message}")
        print(f"Intent  : {label}")

    if args.evaluate:
        with open(args.config, encoding="utf-8") as f:
            cfg = yaml.safe_load(f)
        evaluate(classifier, cfg["data"]["test_file"])

    if args.interactive:
        interactive_mode(classifier)

    if not args.message and not args.evaluate and not args.interactive:
        print("\nNo action specified. Use --message, --evaluate, or --interactive.")
        print("Example:")
        print('  python scripts/inference.py --config configs/inference.yaml \\')
        print('      --message "I want to cancel my bank transfer"')


if __name__ == "__main__":
    main()


Overwriting scripts/inference.py


## Step 3: Data Preparation

In [27]:
# ============================================================
# Preprocess BANKING77 dataset
# ============================================================
!python scripts/preprocess_data.py \
    --train_samples_per_class 40 \
    --test_samples_per_class 10 \
    --output_dir sample_data \
    --seed 42

import pandas as pd, json
print('\n--- Sample training data ---')
train_df = pd.read_csv('sample_data/train.csv')
print(f'Shape: {train_df.shape}')
display(train_df.head(10))
print(f'\nLabel distribution (first 10):')
print(train_df['label_text'].value_counts().head(10))
print(f'\n--- Label map (first 10) ---')
with open('sample_data/label_map.json') as f:
    lm = json.load(f)
for k, v in list(lm['id2label'].items())[:10]:
    print(f'  {k}: {v}')


Loading BANKING77 dataset from HuggingFace …
Using the latest cached version of the dataset since PolyAI/banking77 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /root/.cache/huggingface/datasets/PolyAI___banking77/default/0.0.0/689fcc406cf47a5fbe15f09393be5f206a009fcc (last modified on Mon Apr 13 05:35:01 2026).
  Loaded via default method.
/content/banking-intent-unsloth/scripts/preprocess_data.py:122: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/content/banking-intent-unsloth/scripts/preprocess_data.py:134: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future ver

,text,label,label_text
0,does the fee for exchange change?,17,card_payment_wrong_exchange_rate
1,i topped up and now my money is not there anym...,61,top_up_reverted
2,my money i had was gone and i could not get gas!,62,topping_up_by_card
3,what other currencies are available to change to?,33,exchange_via_app
4,how can i report that my card was stolen? i ma...,41,lost_or_stolen_card
5,my top up may have been reverted,61,top_up_reverted
6,can i receive transfers from my employer this ...,50,receiving_money
7,"i got some cash at an atm earlier, but now app...",75,wrong_amount_of_cash_received
8,need help to get my card to work.,14,card_not_working
9,why was i charged $1 in a transaction?,34,extra_charge_on_statement



Label distribution (first 10):
label_text
card_payment_wrong_exchange_rate    40
top_up_reverted                     40
topping_up_by_card                  40
exchange_via_app                    40
lost_or_stolen_card                 40
receiving_money                     40
wrong_amount_of_cash_received       40
card_not_working                    40
extra_charge_on_statement           40
declined_card_payment               40
Name: count, dtype: int64

--- Label map (first 10) ---
  0: activate_my_card
  1: age_limit
  2: apple_pay_or_google_pay
  3: atm_support
  4: automatic_top_up
  5: balance_not_updated_after_bank_transfer
  6: balance_not_updated_after_cheque_or_cash_deposit
  7: beneficiary_not_allowed
  8: cancel_transfer
  9: card_about_to_expire


## Step 4: Fine-Tuning with Unsloth

This cell fine-tunes **LLaMA 3.2 3B** with QLoRA. On an H100 this takes roughly **10–20 minutes**.

In [28]:
# ============================================================
# Fine-tune with Unsloth
# ============================================================
!python scripts/train.py --config configs/train.yaml


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Training samples : 3075
Test samples     : 770
Num labels       : 77

Loading model: unsloth/Llama-3.2-3B-Instruct-bnb-4bit …
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 254/254 [00:00<00:00, 644.32it/s]
Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 202

## Step 5: Inference & Evaluation

In [29]:
# ============================================================
# Inference demos (loads model ONCE, runs all demos)
# ============================================================
from scripts.inference import IntentClassification

classifier = IntentClassification('configs/inference.yaml')

demo_messages = [
    "I want to activate my new card",
    "Why was I charged an extra fee on my statement?",
    "My card payment is still pending after 3 days",
    "How do I change my PIN number?",
    "I lost my phone and need to secure my account",
    "Can I transfer money to someone in another country?",
    "I need a replacement card",
    "What is the exchange rate for USD to EUR?",
]

print("=" * 60)
print("  INFERENCE DEMOS")
print("=" * 60)

for msg in demo_messages:
    label = classifier(msg)
    print(f"\nMessage : {msg}")
    print(f"Intent  : {label}")

print("\n" + "=" * 60)


Loading model from: saved_model …
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Loaded 77 intent labels.
  INFERENCE DEMOS


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

...


Message : My card payment is still pending after 3 days
Intent  : pending_card_payment

Message : How do I change my PIN number?
Intent  : change_pin


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Message : I lost my phone and need to secure my account
Intent  : lost_or_stolen_phone

Message : Can I transfer money to someone in another country?
Intent  : receiving_money


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Message : I need a replacement card
Intent  : card_swallowed

Message : What is the exchange rate for USD to EUR?
Intent  : exchange_rate



In [30]:
# ============================================================
# Full test-set evaluation (reuses the model loaded above)
# ============================================================
from scripts.inference import evaluate
import yaml

with open('configs/inference.yaml') as f:
    cfg = yaml.safe_load(f)

evaluate(classifier, cfg['data']['test_file'])


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Running evaluation on 770 test samples …



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=6

  [50/770]  acc=0.8600  elapsed=8.6s


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  [750/770]  acc=0.8853  elapsed=121.0s


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


Test Accuracy: 0.8844  (681/770)

Classification Report:
                                                  precision    recall  f1-score   support

                           Refund_not_showing_up       0.90      0.90      0.90        10
                                activate_my_card       1.00      0.90      0.95        10
                                       age_limit       1.00      1.00      1.00        10
                         apple_pay_or_google_pay       0.91      1.00      0.95        10
                                     atm_support       1.00      1.00      1.00        10
                                automatic_top_up       0.90      0.90      0.90        10
         balance_not_updated_after_bank_transfer       1.00      1.00      1.00        10
balance_not_updated_after_cheque_or_cash_deposit       1.00      0.70      0.82        10
                         beneficiary_not_allowed       0.71      0.50      0.59        10
                                 cancel_t

0.8844155844155844

## Step 6: Push to GitHub

This cell creates the GitHub repo (if needed) and pushes all code + data.

In [31]:
# ============================================================
# Create GitHub repo & push everything
# ============================================================
import requests

try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_PAT')
    print('GitHub PAT loaded from Colab Secrets')
except Exception:
    TOKEN = input('Enter your GitHub Personal Access Token: ')

GITHUB_USER = 'ledinhminhquan'
REPO_NAME   = 'banking-intent-unsloth'

headers = {'Authorization': f'token {TOKEN}',
           'Accept': 'application/vnd.github.v3+json'}
resp = requests.post(
    'https://api.github.com/user/repos',
    headers=headers,
    json={'name': REPO_NAME,
          'description': 'Banking77 intent classification with Unsloth (CSC15012)',
          'private': False}
)
if resp.status_code == 201:
    print(f'Repository created: https://github.com/{GITHUB_USER}/{REPO_NAME}')
elif resp.status_code == 422:
    print(f'Repository already exists: https://github.com/{GITHUB_USER}/{REPO_NAME}')
else:
    print(f'Repo API response: {resp.status_code} - {resp.json()}')

!git init
!git config user.email 'ledinhminhquan@users.noreply.github.com'
!git config user.name  'ledinhminhquan'
!git add .
!git status
!git commit -m "Complete Banking77 intent classification with Unsloth" || echo "Nothing new to commit"
!git branch -M main

remote_url = f'https://{TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
!git remote remove origin 2>/dev/null
!git remote add origin {remote_url}
!git push -u origin main --force

print(f'\nAll code pushed to GitHub!')
print(f'https://github.com/{GITHUB_USER}/{REPO_NAME}')


GitHub PAT loaded from Colab Secrets
Repository created: https://github.com/ledinhminhquan/banking-intent-unsloth
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/banking-intent-unsloth/.git/
On branch master

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .gitignore
	new file:   README.md
	new file:   configs/inference.yaml
	new file:   configs/train.yaml
	new file:   huggingface_tokenizers_cache/models--unsloth--Llama-3.2-3B-Instruct-bnb-4bit/.no_exist/bb1d31

## All Done!

### Completed automatically:
- Project structure created
- BANKING77 data preprocessed (3,080 train / 770 test)
- LLaMA 3.2 3B fine-tuned with QLoRA via Unsloth
- Model evaluated on test set
- Inference demos run
- Everything pushed to GitHub


---
**GitHub:** https://github.com/ledinhminhquan/banking-intent-unsloth
